# Sentimento em avaliações de e-commerce (PT-BR): fine-tuning de BERTimbau e Qwen2.5 + LoRA

Notebook de treino do projeto [avaliacoes-ptbr-llm](https://github.com/rafantavares/avaliacoes-ptbr-llm).

**Tarefa:** classificar avaliações reais da Olist em *negativo* (1–2 estrelas), *neutro* (3) e *positivo* (4–5) a partir do texto.

**Comparação** no mesmo conjunto de teste (5.259 avaliações, sem duplicatas vazando entre partições):

| Abordagem | Onde roda |
|---|---|
| TF-IDF + regressão logística | CPU, segundos |
| BERTimbau base — ajuste completo | GPU |
| Qwen2.5-0.5B-Instruct — LoRA (~1% dos parâmetros) | GPU |
| LLM zero-shot via API | fora deste notebook (`avaliacoes.llm_zero_shot`) |

**Para rodar:** *Accelerator* → GPU T4 x2 ou P100; *Internet* ligada. O dataset `olistbr/brazilian-ecommerce` é usado como *Input* se estiver anexado, ou baixado automaticamente.

In [ ]:
!pip install -q "avaliacoes-ptbr-llm[treino] @ git+https://github.com/rafantavares/avaliacoes-ptbr-llm"

## 1. Dados — mesma partição usada fora do Kaggle

In [ ]:
import glob, json
import pandas as pd
from avaliacoes.dados import preparar

ARQUIVO = "olist_order_reviews_dataset.csv"
encontrados = glob.glob(f"/kaggle/input/**/{ARQUIVO}", recursive=True)
if encontrados:
    csv = encontrados[0]
else:  # dataset não anexado como Input: baixa direto do Kaggle
    import kagglehub
    csv = f"{kagglehub.dataset_download('olistbr/brazilian-ecommerce')}/{ARQUIVO}"
info = preparar(csv, "data/processed")
print(info)

# Impressão digital da partição de teste gerada localmente: garante que todos os
# modelos do projeto foram avaliados exatamente nas mesmas avaliações.
IMPRESSAO_ESPERADA = "6595bec2f909df91"
assert info["impressao_teste"] == IMPRESSAO_ESPERADA, "partição de teste diferente da do repositório"

## 2. Baseline clássico

In [ ]:
from avaliacoes.baseline import treinar_e_avaliar as baseline
resultados = {"tfidf_logreg": baseline()}
print(json.dumps(resultados["tfidf_logreg"]["f1_por_classe"], indent=2))

## 3. BERTimbau base (ajuste completo)

In [ ]:
import gc, torch
from avaliacoes.finetune import BERTIMBAU, QWEN_LORA, treinar_e_avaliar

resultados["bertimbau_base"] = treinar_e_avaliar(BERTIMBAU)
gc.collect(); torch.cuda.empty_cache()

## 4. Qwen2.5-0.5B-Instruct + LoRA

In [ ]:
resultados["qwen2.5_0.5b_lora"] = treinar_e_avaliar(QWEN_LORA)

## 5. Comparação

In [ ]:
colunas = ["f1_macro", "acuracia", "ms_por_amostra", "segundos_treino", "hardware"]
tabela = pd.DataFrame(resultados).T[colunas]
for r in resultados.values():
    for classe, f1 in r["f1_por_classe"].items():
        tabela.loc[r["modelo"], f"f1_{classe}"] = f1
tabela.sort_values("f1_macro", ascending=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, eixos = plt.subplots(1, len(resultados), figsize=(5 * len(resultados), 4))
for eixo, r in zip(eixos, resultados.values()):
    m = np.array(r["matriz_confusao"]["valores"])
    m = m / m.sum(axis=1, keepdims=True)
    eixo.imshow(m, cmap="Blues", vmin=0, vmax=1)
    classes = r["matriz_confusao"]["classes"]
    eixo.set_xticks(range(3), classes); eixo.set_yticks(range(3), classes)
    eixo.set_xlabel("previsto"); eixo.set_ylabel("verdadeiro")
    eixo.set_title(f"{r['modelo']}\nF1-macro {r['f1_macro']:.3f}")
    for i in range(3):
        for j in range(3):
            eixo.text(j, i, f"{m[i, j]:.0%}", ha="center", va="center",
                      color="white" if m[i, j] > 0.5 else "black")
plt.tight_layout()
plt.savefig("results/matrizes_confusao.png", dpi=120)

## 6. Publicar no Hugging Face (opcional)

Requer um *secret* do Kaggle chamado `HF_TOKEN` (token com permissão de escrita).

In [ ]:
from pathlib import Path
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

USUARIO_HF = "rafantavares"
api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))

CARTAO = '''---
language: pt
license: cc-by-nc-sa-4.0
library_name: {biblioteca}
base_model: {base}
datasets: [olistbr/brazilian-ecommerce]
pipeline_tag: text-classification
tags: [sentiment-analysis, portuguese, e-commerce]
metrics: [f1]
---

# {nome}: sentimento de avaliações de e-commerce em PT-BR

Classifica avaliações de clientes em **negativo**, **neutro** ou **positivo**.
Treinado com as avaliações públicas da Olist (1–2 estrelas = negativo, 3 = neutro, 4–5 = positivo).

| Métrica (teste, {n} avaliações) | Valor |
|---|---|
| F1-macro | {f1_macro} |
| Acurácia | {acuracia} |
| F1 negativo / neutro / positivo | {f1_neg} / {f1_neu} / {f1_pos} |

Comparação completa com baseline clássico e LLM zero-shot, código e limitações:
https://github.com/rafantavares/avaliacoes-ptbr-llm

**Limitações:** a classe *neutro* é a mais difícil (é rara e o texto de quem dá 3 estrelas costuma ser ambíguo).
O rótulo vem da nota, não de anotação humana do texto. Licença não comercial, herdada do dataset.
'''

for nome, repo, biblioteca in [
    ("bertimbau_base", "bertimbau-sentimento-avaliacoes-ptbr", "transformers"),
    ("qwen2.5_0.5b_lora", "qwen2.5-0.5b-lora-sentimento-avaliacoes-ptbr", "peft"),
]:
    r = resultados[nome]
    pasta = Path("models") / nome / "final"
    (pasta / "README.md").write_text(CARTAO.format(
        nome=repo, base=r["modelo_base"], biblioteca=biblioteca, n=r["n_teste"],
        f1_macro=r["f1_macro"], acuracia=r["acuracia"],
        f1_neg=r["f1_por_classe"]["negativo"], f1_neu=r["f1_por_classe"]["neutro"],
        f1_pos=r["f1_por_classe"]["positivo"],
    ), encoding="utf-8")
    repo_id = f"{USUARIO_HF}/{repo}"
    api.create_repo(repo_id, exist_ok=True)
    api.upload_folder(folder_path=str(pasta), repo_id=repo_id)
    print("publicado:", f"https://huggingface.co/{repo_id}")

Os arquivos `results/*.json` e `results/matrizes_confusao.png` ficam na aba *Output* — são os mesmos que vão para o repositório.